# Day 10 — Pandas I: Selection & Aggregation
## `pandas_business_qs` Notebook

**What we'll cover:**
1. Setup & dataset creation
2. Exercises: `loc`/`iloc`, Boolean filtering, `groupby().agg()`, `merge`, `sort_values`
3. 15 Business questions on a real-looking sales dataset
4. Self-review & reflection

---


## 0. Imports & settings

In [4]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("✅ Ready — pandas", pd.__version__)


✅ Ready — pandas 2.3.3


## 1. Build the Sales Dataset

We generate 500 synthetic sales orders.
Think of this as data exported from an e-commerce system.

| Column | What it means |
|---|---|
| `order_id` | Unique ID for each order |
| `order_date` | Date the order was placed |
| `customer_id` | Customer number |
| `product` | Product name |
| `category` | Product category (Electronics, Furniture, Stationery) |
| `region` | Sales region (North, South, East, West) |
| `quantity` | Number of units ordered |
| `unit_price` | Price per unit (USD) |
| `discount` | Discount fraction, e.g. 0.10 = 10% off |
| `sales_rep` | Name of the sales representative |
| `revenue` | `quantity × unit_price × (1 − discount)` |


In [5]:
np.random.seed(42)
N = 500

# Product catalogue: name → (category, price)
products = {
    'Laptop Pro':     ('Electronics', 1200),
    'Wireless Mouse': ('Electronics',   35),
    'USB-C Hub':      ('Electronics',   55),
    'Desk Chair':     ('Furniture',    450),
    'Standing Desk':  ('Furniture',    800),
    'Bookshelf':      ('Furniture',    150),
    'Notebook Set':   ('Stationery',    18),
    'Whiteboard':     ('Stationery',    95),
    'Pen Pack':       ('Stationery',     8),
    'Monitor 27"':    ('Electronics',  380),
    'Headphones':     ('Electronics',  120),
    'Filing Cabinet': ('Furniture',    200),
}

regions    = ['North', 'South', 'East', 'West']
sales_reps = ['Alice', 'Bob', 'Carol', 'David', 'Eva']

chosen_products = np.random.choice(list(products.keys()), N)

orders = pd.DataFrame({
    'order_id':    range(1001, 1001 + N),
    'order_date':  pd.to_datetime(
                       np.random.choice(
                           pd.date_range('2023-01-01', '2023-12-31').astype(str), N)),
    'customer_id': np.random.randint(200, 300, N),
    'product':     chosen_products,
    'category':    [products[p][0] for p in chosen_products],
    'region':      np.random.choice(regions, N),
    'quantity':    np.random.randint(1, 11, N),
    'unit_price':  [products[p][1] for p in chosen_products],
    'discount':    np.round(
                       np.random.choice([0, 0.05, 0.10, 0.20, 0.30], N,
                                        p=[0.5, 0.2, 0.15, 0.1, 0.05]), 2),
    'sales_rep':   np.random.choice(sales_reps, N),
})

orders['revenue'] = (
    orders['quantity'] * orders['unit_price'] * (1 - orders['discount'])
).round(2)

# Introduce 5 missing values to practice edge cases
orders.loc[[10, 55, 120, 300, 450], 'discount'] = np.nan

print(f"Dataset shape: {orders.shape[0]} rows × {orders.shape[1]} columns")
orders.head(8)


Dataset shape: 500 rows × 11 columns


,order_id,order_date,customer_id,product,category,region,quantity,unit_price,discount,sales_rep,revenue
0,1001,2023-04-04,293,Notebook Set,Stationery,South,10,18,0.10,Alice,162.00
1,1002,2023-08-26,250,Desk Chair,Furniture,North,3,450,0.00,David,"1,350.00"
2,1003,2023-07-23,261,Headphones,Electronics,East,4,120,0.05,Bob,456.00
3,1004,2023-08-06,256,Whiteboard,Stationery,North,9,95,0.00,Carol,855.00
4,1005,2023-12-07,265,Standing Desk,Furniture,East,7,800,0.00,David,"5,600.00"
5,1006,2023-02-08,278,Notebook Set,Stationery,West,1,18,0.10,David,16.20
6,1007,2023-04-10,274,"Monitor 27""",Electronics,South,4,380,0.20,Alice,"1,216.00"
7,1008,2023-10-16,207,USB-C Hub,Electronics,West,4,55,0.05,Carol,209.00


### 1a. Always inspect first

In [6]:
# info() shows column types + null counts at a glance
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     500 non-null    int64         
 1   order_date   500 non-null    datetime64[ns]
 2   customer_id  500 non-null    int32         
 3   product      500 non-null    object        
 4   category     500 non-null    object        
 5   region       500 non-null    object        
 6   quantity     500 non-null    int32         
 7   unit_price   500 non-null    int64         
 8   discount     495 non-null    float64       
 9   sales_rep    500 non-null    object        
 10  revenue      500 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int32(2), int64(2), object(4)
memory usage: 39.2+ KB


In [7]:
# describe() gives quick numeric stats
orders.describe()


,order_id,order_date,customer_id,quantity,unit_price,discount,revenue
count,500.00,500,500.00,500.00,500.00,495.00,500.00
mean,"1,250.50",2023-06-28 11:34:04.800000256,247.84,5.53,308.33,0.06,"1,639.59"
min,"1,001.00",2023-01-01 00:00:00,200.00,1.00,8.00,0.00,5.60
25%,"1,125.75",2023-03-31 18:00:00,224.75,3.00,55.00,0.00,171.38
50%,"1,250.50",2023-07-01 12:00:00,246.00,6.00,150.00,0.00,604.00
75%,"1,375.25",2023-09-17 12:00:00,272.00,8.00,450.00,0.10,"1,900.00"
max,"1,500.00",2023-12-31 00:00:00,299.00,10.00,"1,200.00",0.30,"12,000.00"
std,144.48,NaN,28.73,2.89,370.38,0.08,"2,441.86"


---
## 2. Exercises

### Exercise A — `loc` and `iloc`

Your DataFrame is like a spreadsheet table.
- `loc` → "give me rows/columns by their **name** (label)"
- `iloc` → "give me rows/columns by their **position number** (0, 1, 2...)"

```
         order_id   product   revenue
index 0    1001    Laptop Pro   1200    ← loc[0] or iloc[0] (same here)
index 1    1002    Pen Pack       8    ← loc[1] or iloc[1]
index 2    1003    Desk Chair   450    ← loc[2] or iloc[2]
```


In [8]:
# loc: pick rows 0–4, specific columns by NAME
orders.loc[0:4, ['order_id', 'product', 'quantity', 'revenue']]

,order_id,product,quantity,revenue
0,1001,Notebook Set,10,162.00
1,1002,Desk Chair,3,"1,350.00"
2,1003,Headphones,4,456.00
3,1004,Whiteboard,9,855.00
4,1005,Standing Desk,7,"5,600.00"


In [9]:
# iloc: pick rows 0–4, columns at positions 0,3,4,10 by NUMBER
#   position 0=order_id, 3=product, 4=category, 10=revenue
orders.iloc[0:5, [0, 3, 4, 10]]

,order_id,product,category,revenue
0,1001,Notebook Set,Stationery,162.00
1,1002,Desk Chair,Furniture,"1,350.00"
2,1003,Headphones,Electronics,456.00
3,1004,Whiteboard,Stationery,855.00
4,1005,Standing Desk,Furniture,"5,600.00"


In [10]:
# iloc: last 3 rows, all columns
orders.iloc[-3:, :]

,order_id,order_date,customer_id,product,category,region,quantity,unit_price,discount,sales_rep,revenue
497,1498,2023-01-04,248,Filing Cabinet,Furniture,West,5,200,0.00,Bob,"1,000.00"
498,1499,2023-12-01,260,Standing Desk,Furniture,North,4,800,0.30,Alice,"2,240.00"
499,1500,2023-05-14,216,Laptop Pro,Electronics,West,8,1200,0.00,Carol,"9,600.00"


### Exercise B — Boolean Filtering

**Plain English:**
Write a condition → pandas checks every row → returns True/False.
Wrap it in `orders[...]` to keep only the True rows.

```python
orders['region'] == 'North'   # → [True, False, True, False, ...]
orders[orders['region'] == 'North']   # → only North rows
```

Combine conditions:
- `&` means AND (both must be True)
- `|` means OR  (at least one must be True)
- Always wrap each condition in `()`  ← easy to forget!


In [11]:
# Single condition: only Electronics
electronics = orders[orders['category'] == 'Electronics']
print(f"Electronics rows: {len(electronics)}")
electronics[['product', 'quantity', 'revenue']].head(4)


Electronics rows: 201


,product,quantity,revenue
2,Headphones,4,456.00
6,"Monitor 27""",4,"1,216.00"
7,USB-C Hub,4,209.00
9,Headphones,6,720.00


In [12]:
# AND condition: Electronics with revenue > 1000
big_elec = orders[
    (orders['category'] == 'Electronics') &
    (orders['revenue'] > 1000)
]
print(f"High-value electronics: {len(big_elec)}")
big_elec[['product', 'region', 'quantity', 'revenue']].head(5)


High-value electronics: 89


,product,region,quantity,revenue
6,"Monitor 27""",South,4,"1,216.00"
10,Headphones,West,9,"1,080.00"
26,Laptop Pro,South,6,"7,200.00"
28,"Monitor 27""",North,8,"3,040.00"
32,Laptop Pro,South,9,"10,260.00"


In [13]:
# isin() is cleaner than multiple OR conditions
north_south = orders[orders['region'].isin(['North', 'South'])]
print(f"North + South orders: {len(north_south)}")

North + South orders: 252


In [14]:
# Edge case: find rows with missing discount
missing = orders[orders['discount'].isnull()]
print(f"Orders with missing discount: {len(missing)}")
missing[['order_id', 'product', 'discount']]

Orders with missing discount: 5


,order_id,product,discount
10,1011,Headphones,NaN
55,1056,"Monitor 27""",NaN
120,1121,Pen Pack,NaN
300,1301,USB-C Hub,NaN
450,1451,Notebook Set,NaN


### Exercise C — `groupby().agg()`

Like a pivot table. You say:
1. *"Split my data by column X"*  →  `groupby('X')`
2. *"For each group, compute..."*  →  `.agg(new_col=('col', 'function'))`

```
Before groupby:          After groupby('region').sum():
region  revenue          region   revenue
North   100              North    300
North   200              South    150
South   150
```

Common functions: `'sum'`, `'mean'`, `'count'`, `'max'`, `'min'`, `'nunique'`


In [15]:
# Revenue by region (single aggregation)
(
    orders
    .groupby('region')['revenue']
    .sum()
    .reset_index(name='total_revenue')   # turns the grouped index back into a column
    .sort_values('total_revenue', ascending=False)
)


,region,total_revenue
1,North,"252,265.70"
0,East,"202,595.65"
2,South,"189,877.20"
3,West,"175,055.15"


In [16]:
# Multiple stats at once with agg()
(
    orders
    .groupby('category')
    .agg(
        total_revenue=('revenue',  'sum'),    # sum of revenue
        order_count  =('order_id', 'count'),  # count of orders
        avg_revenue  =('revenue',  'mean'),   # average order value
        max_order    =('revenue',  'max'),    # biggest single order
    )
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)

,category,total_revenue,order_count,avg_revenue,max_order
0,Electronics,"463,215.25",201,"2,304.55","12,000.00"
1,Furniture,"330,237.50",171,"1,931.21","8,000.00"
2,Stationery,"26,340.95",128,205.79,950.00


### Exercise D — `merge` / `join`

Exactly like SQL JOIN — combine two tables using a shared column (the "key").

```
orders:                  product_info:
order_id | product        product    | supplier
1001     | Laptop    +    Laptop     | TechCorp    →  order_id | product | supplier
1002     | Chair          Chair      | FurniCo         1001    | Laptop  | TechCorp
                                                       1002    | Chair   | FurniCo
```

`how='left'` means: keep ALL rows from the left table.
If the right table has no match, fill with NaN.


In [17]:
# Create a small product lookup table
np.random.seed(99)
product_info = pd.DataFrame({
    'product':        list(orders['product'].unique()),
    'supplier':       np.random.choice(
                          ['TechCorp','FurniCo','OfficePro','MegaSupply'],
                          orders['product'].nunique()),
    'warranty_years': np.random.choice([1, 2, 3], orders['product'].nunique()),
})
product_info


,product,supplier,warranty_years
0,Notebook Set,FurniCo,3
1,Desk Chair,MegaSupply,2
2,Headphones,FurniCo,2
3,Whiteboard,TechCorp,1
4,Standing Desk,FurniCo,2
5,"Monitor 27""",TechCorp,3
6,USB-C Hub,OfficePro,1
7,Bookshelf,TechCorp,3
8,Wireless Mouse,FurniCo,1
9,Filing Cabinet,TechCorp,1


In [18]:
# Left join: all orders kept, product_info columns added
enriched = orders.merge(product_info, on='product', how='left')
print(f"Shape before merge: {orders.shape}")
print(f"Shape after  merge: {enriched.shape}")
enriched[['order_id', 'product', 'supplier', 'warranty_years', 'revenue']].head(5)

Shape before merge: (500, 11)
Shape after  merge: (500, 13)


,order_id,product,supplier,warranty_years,revenue
0,1001,Notebook Set,FurniCo,3,162.00
1,1002,Desk Chair,MegaSupply,2,"1,350.00"
2,1003,Headphones,FurniCo,2,456.00
3,1004,Whiteboard,TechCorp,1,855.00
4,1005,Standing Desk,FurniCo,2,"5,600.00"


### Exercise E — `sort_values`

Sort rows by one or more columns.
- `ascending=False` → biggest first (most common for "top N" reports)
- `ascending=True`  → smallest first (default)


In [19]:
# Top 10 orders by revenue
orders.sort_values('revenue', ascending=False).head(10)[
    ['order_id', 'product', 'region', 'quantity', 'revenue']
]

,order_id,product,region,quantity,revenue
310,1311,Laptop Pro,South,10,"12,000.00"
208,1209,Laptop Pro,North,10,"12,000.00"
183,1184,Laptop Pro,West,10,"12,000.00"
336,1337,Laptop Pro,South,9,"10,800.00"
400,1401,Laptop Pro,West,9,"10,800.00"
274,1275,Laptop Pro,North,9,"10,800.00"
219,1220,Laptop Pro,South,10,"10,800.00"
184,1185,Laptop Pro,West,9,"10,800.00"
67,1068,Laptop Pro,West,9,"10,800.00"
32,1033,Laptop Pro,South,9,"10,260.00"


In [20]:
# Sort by multiple columns: region A→Z, then revenue biggest first
orders.sort_values(
    ['region', 'revenue'],
    ascending=[True, False]    # True for region, False for revenue
).head(8)[['order_id', 'region', 'product', 'revenue']]

,order_id,region,product,revenue
380,1381,East,Laptop Pro,"10,260.00"
361,1362,East,Laptop Pro,"9,600.00"
251,1252,East,Laptop Pro,"8,400.00"
479,1480,East,Laptop Pro,"7,680.00"
356,1357,East,Standing Desk,"7,200.00"
104,1105,East,Laptop Pro,"6,000.00"
215,1216,East,Laptop Pro,"5,700.00"
4,1005,East,Standing Desk,"5,600.00"


---
## 3. Task — 15 Business Questions

---


### Q1 — What are the top 5 best-selling products by total revenue?

*Pandas skill used: groupby + sum + sort_values*

In [21]:
(
    orders
    .groupby('product')['revenue']
    .sum()
    .reset_index(name='total_revenue')
    .sort_values('total_revenue', ascending=False)
    .head(5)
)

,product,total_revenue
4,Laptop Pro,"337,200.00"
8,Standing Desk,"155,160.00"
1,Desk Chair,"88,897.50"
5,"Monitor 27""","88,540.00"
2,Filing Cabinet,"57,710.00"


### Q2 — What is the total revenue for each region?

*Pandas skill used: groupby + sum*

In [22]:
(
    orders
    .groupby('region')['revenue']
    .sum()
    .reset_index(name='total_revenue')
    .sort_values('total_revenue', ascending=False)
)

,region,total_revenue
1,North,"252,265.70"
0,East,"202,595.65"
2,South,"189,877.20"
3,West,"175,055.15"


### Q3 — Which product category generates the most orders?

*Pandas skill used: groupby + count*

In [23]:
(
    orders
    .groupby('category')['order_id']
    .count()
    .reset_index(name='order_count')
    .sort_values('order_count', ascending=False)
)

,category,order_count
0,Electronics,201
1,Furniture,171
2,Stationery,128


### Q4 — Which sales rep has the highest total revenue?

*Pandas skill used: groupby + sum*

In [24]:
(
    orders
    .groupby('sales_rep')['revenue']
    .sum()
    .reset_index(name='total_revenue')
    .sort_values('total_revenue', ascending=False)
)

,sales_rep,total_revenue
2,Carol,"226,250.90"
0,Alice,"190,578.30"
4,Eva,"144,602.20"
3,David,"132,465.65"
1,Bob,"125,896.65"


### Q5 — What is the average order value per region?

*Pandas skill used: groupby + mean*

In [25]:
(
    orders
    .groupby('region')['revenue']
    .mean()
    .reset_index(name='avg_order_value')
    .sort_values('avg_order_value', ascending=False)
)

,region,avg_order_value
1,North,"1,896.73"
2,South,"1,595.61"
0,East,"1,570.51"
3,West,"1,471.05"


### Q6 — How many orders had a discount applied?

*Pandas skill used: Boolean filter + isnull edge case*

In [26]:
# Edge case: some discount values are NaN — treat them as no discount
orders_with_discount    = orders[orders['discount'].fillna(0) > 0]
orders_without_discount = orders[orders['discount'].fillna(0) == 0]
orders_missing_discount = orders[orders['discount'].isnull()]

print(f"With discount:    {len(orders_with_discount):>4} ({len(orders_with_discount)/len(orders)*100:.1f}%)")
print(f"Without discount: {len(orders_without_discount):>4} ({len(orders_without_discount)/len(orders)*100:.1f}%)")
print(f"Missing (NaN):    {len(orders_missing_discount):>4}")

With discount:     242 (48.4%)
Without discount:  258 (51.6%)
Missing (NaN):       5


### Q7 — What are the monthly revenue trends throughout 2023?

*Pandas skill used: dt accessor + groupby + sum*

In [27]:
monthly = (
    orders
    .assign(month=orders['order_date'].dt.to_period('M'))  # extract year-month
    .groupby('month')['revenue']
    .sum()
    .reset_index(name='monthly_revenue')
)
monthly['month'] = monthly['month'].astype(str)
print("Monthly revenue 2023:")
monthly

Monthly revenue 2023:


,month,monthly_revenue
0,2023-01,"85,677.45"
1,2023-02,"72,420.75"
2,2023-03,"43,816.30"
3,2023-04,"53,005.55"
4,2023-05,"86,265.40"
5,2023-06,"67,965.00"
6,2023-07,"50,207.60"
7,2023-08,"66,916.85"
8,2023-09,"59,126.80"
9,2023-10,"63,556.10"


### Q8 — Which product has the highest average order quantity?

*Pandas skill used: groupby + mean*

In [28]:
(
    orders
    .groupby('product')['quantity']
    .mean()
    .reset_index(name='avg_quantity')
    .sort_values('avg_quantity', ascending=False)
)

,product,avg_quantity
5,"Monitor 27""",6.15
6,Notebook Set,5.87
4,Laptop Pro,5.73
1,Desk Chair,5.58
11,Wireless Mouse,5.55
0,Bookshelf,5.45
7,Pen Pack,5.44
2,Filing Cabinet,5.43
9,USB-C Hub,5.34
3,Headphones,5.33


### Q9 — What percentage of total revenue comes from each category?

*Pandas skill used: groupby + sum + percentage*

In [29]:
cat_rev   = orders.groupby('category')['revenue'].sum()
grand_tot = cat_rev.sum()

(
    cat_rev
    .reset_index(name='revenue')
    .assign(pct_of_total=lambda df: (df['revenue'] / grand_tot * 100).round(2))
    .sort_values('pct_of_total', ascending=False)
)

,category,revenue,pct_of_total
0,Electronics,"463,215.25",56.50
1,Furniture,"330,237.50",40.28
2,Stationery,"26,340.95",3.21


### Q10 — Which region offers the highest average discount?

*Pandas skill used: dropna edge case + groupby + mean*

In [30]:
# dropna(subset=['discount']) removes the 5 rows where discount is NaN
q10 = (
    orders.dropna(subset=['discount'])
    .groupby('region')['discount']
    .mean()
    .reset_index(name='avg_discount')
    .sort_values('avg_discount', ascending=False)
)
q10['avg_discount_pct'] = (q10['avg_discount'] * 100).round(2)
q10

,region,avg_discount,avg_discount_pct
1,North,0.06,6.31
0,East,0.06,6.21
2,South,0.06,5.67
3,West,0.05,4.70


### Q11 — Who are the top 10 customers by total spending?

*Pandas skill used: groupby + sum + head*

In [31]:
(
    orders
    .groupby('customer_id')['revenue']
    .sum()
    .reset_index(name='total_spent')
    .sort_values('total_spent', ascending=False)
    .head(10)
)

,customer_id,total_spent
16,216,"29,268.50"
4,204,"22,288.00"
34,234,"20,300.25"
37,237,"20,185.50"
53,253,"19,480.00"
72,272,"19,162.00"
46,246,"18,499.00"
50,250,"18,267.25"
88,289,"17,991.00"
80,281,"17,730.00"


### Q12 — Revenue breakdown by region AND category

*Pandas skill used: multi-column groupby*

In [32]:
(
    orders
    .groupby(['region', 'category'])['revenue']
    .sum()
    .reset_index(name='revenue')
    .sort_values(['region', 'revenue'], ascending=[True, False])
)

,region,category,revenue
0,East,Electronics,"111,076.50"
1,East,Furniture,"86,907.50"
2,East,Stationery,"4,611.65"
3,North,Electronics,"144,493.50"
4,North,Furniture,"97,672.50"
5,North,Stationery,"10,099.70"
7,South,Furniture,"94,320.00"
6,South,Electronics,"90,002.75"
8,South,Stationery,"5,554.45"
9,West,Electronics,"117,642.50"


### Q13 — Which orders are in the top 10% by revenue (high-value)?

*Pandas skill used: quantile + Boolean filter*

In [33]:
threshold = orders['revenue'].quantile(0.90)  # 90th percentile value
high_value = orders[orders['revenue'] >= threshold].sort_values('revenue', ascending=False)

print(f"Top 10% threshold: ${threshold:,.2f}")
print(f"High-value orders: {len(high_value)}")
high_value[['order_id', 'product', 'region', 'quantity', 'revenue']].head(10)

Top 10% threshold: $4,584.00
High-value orders: 50


,order_id,product,region,quantity,revenue
208,1209,Laptop Pro,North,10,"12,000.00"
183,1184,Laptop Pro,West,10,"12,000.00"
310,1311,Laptop Pro,South,10,"12,000.00"
219,1220,Laptop Pro,South,10,"10,800.00"
184,1185,Laptop Pro,West,9,"10,800.00"
67,1068,Laptop Pro,West,9,"10,800.00"
400,1401,Laptop Pro,West,9,"10,800.00"
336,1337,Laptop Pro,South,9,"10,800.00"
274,1275,Laptop Pro,North,9,"10,800.00"
32,1033,Laptop Pro,South,9,"10,260.00"


### Q14 — What is the month-over-month revenue growth?

*Pandas skill used: pct_change on monthly groupby*

In [34]:
monthly_rev = (
    orders
    .assign(month=orders['order_date'].dt.to_period('M'))
    .groupby('month')['revenue']
    .sum()
    .reset_index(name='revenue')
)
monthly_rev['mom_growth_pct'] = (monthly_rev['revenue'].pct_change() * 100).round(2)
monthly_rev['month'] = monthly_rev['month'].astype(str)
# NaN for first row is expected (no previous month to compare)
monthly_rev

,month,revenue,mom_growth_pct
0,2023-01,"85,677.45",NaN
1,2023-02,"72,420.75",-15.47
2,2023-03,"43,816.30",-39.50
3,2023-04,"53,005.55",20.97
4,2023-05,"86,265.40",62.75
5,2023-06,"67,965.00",-21.21
6,2023-07,"50,207.60",-26.13
7,2023-08,"66,916.85",33.28
8,2023-09,"59,126.80",-11.64
9,2023-10,"63,556.10",7.49


### Q15 — Full performance summary per sales rep

*Pandas skill used: multi-metric agg + formatting*

In [35]:
summary = (
    orders.dropna(subset=['discount'])   # drop rows with missing discount
    .groupby('sales_rep')
    .agg(
        total_revenue   =('revenue',  'sum'),
        order_count     =('order_id', 'count'),
        avg_order_value =('revenue',  'mean'),
        avg_discount    =('discount', 'mean'),
        unique_products =('product',  'nunique'),
    )
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)
summary['avg_discount_pct'] = (summary['avg_discount'] * 100).round(1)
summary = summary.drop(columns='avg_discount')
summary[['total_revenue','avg_order_value']] = summary[['total_revenue','avg_order_value']].round(2)
print("Sales rep performance summary:")
summary

Sales rep performance summary:


,sales_rep,total_revenue,order_count,avg_order_value,unique_products,avg_discount_pct
2,Carol,"225,700.90",114,"1,979.83",12,5.70
0,Alice,"186,634.30",102,"1,829.75",12,5.30
4,Eva,"144,571.80",101,"1,431.40",12,7.50
3,David,"131,385.65",104,"1,263.32",12,4.80
1,Bob,"125,896.65",74,"1,701.31",12,5.40
